# Notebook 14 — Testing with pytest

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 14 of 20  
**Prerequisites:** Notebook 13  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

A function without tests is a claim without evidence. In scientific software, the
cost of a silent wrong answer (a bug that returns a plausible but incorrect value)
is higher than the cost of a crash — at least a crash is visible.

pytest is the standard testing framework for scientific Python. This notebook builds
the complete test suite for `compbio_utils` that will be maintained through Module 16.

---
## Step 2 — Intuition

A unit test asks: 'given this specific input, does this function return exactly this
output?' A property-based test asks: 'for any input satisfying these constraints,
does this invariant hold?' Both are useful; start with unit tests.

A good test is:
- **Fast** — milliseconds, not seconds
- **Isolated** — tests one function, not a pipeline
- **Deterministic** — same output every run (use fixed seeds)
- **Descriptive** — the test name explains what it's checking

---
## Step 3 — Biological Background

**Why scientific code needs extra testing rigour:**

A bug in a BLAST score calculation might return a biologically plausible e-value
but identify the wrong gene. A bug in GC content computation that misses lowercase
letters would report wrong values for sequences from databases that store both cases.

Edge cases in biology: empty sequences, sequences of only N's, zero-count genes,
single-sample datasets, negative expression values (after log transform).

---
## Step 4 — Mathematical Explanation

**Test coverage:** for a function $f$ with domain $D$, a test suite covers a fraction
$|T|/|D|$ where $T \subseteq D$ is the set of tested inputs. Because $|D|$ is often
infinite (all possible strings), 100% domain coverage is impossible. Good tests cover:
- Typical inputs (happy path)
- Boundary values (empty, single element, max size)
- Known-incorrect inputs (wrong type, negative count)
- Algebraic properties (complement of complement = identity)

---
## Step 5 — Computational Explanation

**pytest patterns used in this notebook:**

| Pattern | Code | When to use |
|---------|------|-------------|
| Basic assert | `assert f(x) == expected` | Simple equality |
| Float comparison | `np.testing.assert_allclose(result, expected, rtol=1e-6)` | Any floating point |
| Exception testing | `with pytest.raises(ValueError):` | Verifying error handling |
| Parametrize | `@pytest.mark.parametrize("seq,expected", [...])` | Multiple inputs |
| Fixture | `@pytest.fixture` | Shared setup data |
| Mark | `@pytest.mark.slow` | Skip slow tests in CI |

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Write the test file for compbio_utils.sequence
import pathlib

repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists(): break
    repo_root = repo_root.parent

test_content = '''
"""Tests for compbio_utils.sequence."""
import pytest
import numpy as np

from compbio_utils.sequence import gc_content, complement, reverse_complement, sliding_gc


# --- gc_content ---

class TestGcContent:

    @pytest.mark.parametrize("seq,expected", [
        ("ATGC",     0.5),
        ("AAAA",     0.0),
        ("GCGC",     1.0),
        ("atgc",     0.5),   # case-insensitive
        ("ATGCGC",   2/3),
        ("G",        1.0),   # single nucleotide
    ])
    def test_valid(self, seq, expected):
        np.testing.assert_allclose(gc_content(seq), expected, rtol=1e-9)

    def test_empty_raises(self):
        with pytest.raises(ValueError):
            gc_content("")

    def test_wrong_type_raises(self):
        with pytest.raises(TypeError):
            gc_content(42)

    def test_returns_float(self):
        assert isinstance(gc_content("ATGC"), float)


# --- complement ---

class TestComplement:

    @pytest.mark.parametrize("seq,expected", [
        ("ATGC", "TACG"),
        ("GCGC", "CGCG"),
        ("TTTT", "AAAA"),
        ("atgc", "tacg"),   # preserves case
    ])
    def test_valid(self, seq, expected):
        assert complement(seq) == expected

    def test_double_complement_identity(self):
        seq = "ATGCGCATGC"
        assert complement(complement(seq)) == seq

    def test_wrong_type_raises(self):
        with pytest.raises(TypeError):
            complement(None)


# --- reverse_complement ---

class TestReverseComplement:

    @pytest.mark.parametrize("seq,expected", [
        ("ATGC",   "GCAT"),
        ("GAATTC", "GAATTC"),  # EcoRI palindrome
    ])
    def test_valid(self, seq, expected):
        assert reverse_complement(seq) == expected

    def test_double_rc_identity(self):
        seq = "ATGCGCATGC"
        assert reverse_complement(reverse_complement(seq)) == seq


# --- sliding_gc ---

class TestSlidingGc:

    def test_output_length(self):
        seq = "ATGCGCATGC"  # length 10
        result = sliding_gc(seq, window=4)
        assert len(result) == 10 - 4 + 1  # 7 windows

    def test_uniform_sequence(self):
        result = sliding_gc("GCGCGCGCGC", window=4)
        np.testing.assert_allclose(result, [1.0] * 7, rtol=1e-9)

    def test_window_larger_than_seq_raises(self):
        with pytest.raises(ValueError):
            sliding_gc("ATG", window=10)
'''

test_path = repo_root / "utilities" / "compbio_utils" / "tests" / "test_sequence.py"
if not test_path.parent.exists():
    print(f"Tests directory missing: {test_path.parent}")
    print("Create it with: mkdir utilities/compbio_utils/tests")
else:
    test_path.write_text(test_content)
    print(f"Written: {test_path}")

In [ ]:
# Cell 6.2 — Run the test suite
import subprocess

test_dir = repo_root / "utilities" / "compbio_utils" / "tests"
if test_dir.exists():
    result = subprocess.run(
        ["python", "-m", "pytest", str(test_dir), "-v", "--tb=short"],
        capture_output=True, text=True, cwd=repo_root
    )
    print(result.stdout[-3000:])  # last 3000 chars
    if result.stderr:
        print("STDERR:", result.stderr[-500:])
else:
    print("tests/ directory not found — complete the exercise in Cell 6.1 first.")

In [ ]:
# Cell 6.3 — pytest fixture pattern: shared synthetic expression data
# This shows the pattern; the actual fixture goes in conftest.py

conftest_content = '''
"""Shared test fixtures for compbio_utils tests."""
import pytest
import numpy as np


@pytest.fixture
def expression_matrix():
    """Synthetic 5-gene × 4-sample expression matrix with fixed seed."""
    rng = np.random.default_rng(42)
    return rng.exponential(scale=5.0, size=(5, 4))


@pytest.fixture
def dna_sequences():
    """A small collection of test DNA sequences."""
    return {
        "brca1_exon1": "ATGCGATCGATCGATCGATCG",
        "tp53_start":  "ATGGAGGAGCCGCAGTCAGATCCTAGCG",
        "ecori_site":  "GAATTC",
    }
'''

conftest_path = repo_root / "utilities" / "compbio_utils" / "tests" / "conftest.py"
if conftest_path.parent.exists():
    conftest_path.write_text(conftest_content)
    print(f"Written: {conftest_path}")
else:
    print("tests/ directory missing")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Test coverage summary (conceptual, not pytest-cov)
import matplotlib.pyplot as plt

modules = ["sequence.py", "stats.py", "plotting.py", "io.py"]
# Estimated: how many of the expected functions have ≥1 test
tested    = [4, 2, 0, 0]   # ← update these
total_fns = [4, 6, 3, 2]

fig, ax = plt.subplots(figsize=(7, 3))
x = range(len(modules))
ax.bar(x, total_fns, color="lightgray", label="Total functions")
ax.bar(x, tested,    color="steelblue", label="Functions with tests")
ax.set_xticks(x); ax.set_xticklabels(modules)
ax.set_ylabel("Count")
ax.set_title("compbio_utils test coverage — Module 01")
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Add a `TestLogFoldChange` class in `test_stats.py` with at least 5 tests.
2. Install `pytest-cov` and run `pytest --cov=compbio_utils`. Report the coverage percentage.
3. Add a `test_pca_svd_matches_sklearn` test that compares `compbio_utils.stats.pca_svd`
   against `sklearn.decomposition.PCA` on a fixed synthetic dataset.
4. What is the difference between a fixture and a parametrize? When do you prefer each?

---
## Quiz — Active Recall

1. What does `@pytest.mark.parametrize` do? Write an example with 3 test cases.
2. Why should floating-point comparisons use `np.testing.assert_allclose` rather than `==`?
3. What is a pytest fixture? Name one advantage over a `setUp` method.
4. What does `pytest.raises(ValueError)` verify?
5. Name two properties of a good unit test that a function-level doctest cannot provide.

---
## Reflection

**Date completed:** ____________________

> *[Did all tests pass? What was the coverage percentage? Which edge cases caught real bugs?]*

---
*Next: `15_packaging_pyproject.ipynb`*